# ⚽ Mission 12: Soccer Vision Lab — Build an AI Tactical Analyst

## 📖 Mission Story
Every professional soccer club tracks where players move during a match. Today, **Coach Artin**, you will build your own sports analytics system!

Your program will watch a soccer video, track a player frame-by-frame, map their movement onto a pitch, generate a professional heatmap, and consult Gemini AI for elite tactical insights.

---

## 🎯 Learning Objectives
* ✅ Draw pitch visualizations and heatmaps using `mplsoccer`
* ✅ Process soccer movement coordinates from structured CSV data
* ✅ Build a Streamlit video intake interface using `st.file_uploader()`
* ✅ Read and display video frames using OpenCV (`cv2`)
* ✅ Perform player detection and tracking with YOLO (`ultralytics`)
* ✅ Map pixel positions to soccer pitch dimensions ($120 \times 80$ yards)
* ✅ Produce tactical movement heatmaps from computer vision tracking data
* ✅ Prompt Gemini AI as an elite UEFA Pro Tactical Analyst

---

### 🟢 Phase 1: Drawing Professional Soccer Pitches (`mplsoccer`)
We begin by using `mplsoccer` to plot simulated touch data and render dynamic position heatmaps.

In [ ]:
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import numpy as np

# 1. Create a professional soccer pitch layout
pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
fig, ax = pitch.draw(figsize=(10, 7))

# 2. Simulate some ball touch data (X: 0 to 120 yards, Y: 0 to 80 yards)
# First, Let's pretend he is a left-winger who cuts inside toward the box

player = "Artin"
position = "Right Defender"

if position == "Left Winger":
    x_coordinates = [10,15,20,25,30,35,40,45,50,55, 60,65,70,75,80,85,90,95,100,105]
    y_coordinates = [10,12,15,18,20,18,15,12,10,15,18,20,22,25,28,30,35,38,40,42]

elif position == "Striker":
    x_coordinates = [70,75,80,85,90,95,100,102,105,108,110,112,115,108,104,100,95,90,88,110]
    y_coordinates = [35,38,40,42,40,38,35,37,40,42,39,36,40,45,48,50,45,42,38,35]

elif position == "Midfielder":
    x_coordinates = [35,40,45,50,55,60,65,70,60,55,50,45,40,55,65,75,70,60,50,45]
    y_coordinates = [25,30,35,40,45,40,35,30,25,20,25,30,35,45,50,45,40,35,30,25]

elif position == "Right Defender":
    # Modern attacking fullback
    # Defends deep but frequently overlaps on the right wing
    x_coordinates = [15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,105,110]
    y_coordinates = [70,72,68,70,72,74,76,74,72,70,68,70,72,74,76,72,68,65,60,55]

# 3. Plot the touches as a heatmap (2D Histogram)
bin_statistic = pitch.bin_statistic(x_coordinates, y_coordinates, statistic='count', bins=(12, 8))
pcm = pitch.heatmap(bin_statistic, ax=ax, cmap='Reds', edgecolor='#f9f9f9', alpha=0.6)

# 4. Draw the individual touch points on top so he sees how it connects
pitch.scatter(x_coordinates, y_coordinates, c='black', s=50, ax=ax, label='Ball Touches')

plt.title(
    f"{player}'s Soccer Heatmap - {position}",
    fontsize=18,
    fontweight='bold',
    pad=15
)
plt.show()


### 📊 Phase 2: Creating Player Movement Data
Instead of static lists, we structure positions into tabular CSV files for clean data manipulation.

In [ ]:
%%writefile phase2_csv_loader.py
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Generate synthetic trajectory data
data = {
    'frame': list(range(1, 11)),
    'x': [18, 20, 21, 25, 30, 42, 55, 68, 72, 85],
    'y': [52, 53, 54, 50, 48, 45, 40, 38, 35, 30]
}
df = pd.DataFrame(data)
df.to_csv("player_movement.csv", index=False)

# Load CSV and render pitch heatmap
movement_df = pd.read_csv("player_movement.csv")
pitch = Pitch(pitch_type='statsbomb', pitch_color='#101010', line_color='#888888')
fig, ax = pitch.draw(figsize=(10, 7))
pitch.kdeplot(movement_df['x'], movement_df['y'], ax=ax, cmap='magma', fill=True, alpha=0.6)
plt.savefig("phase2_heatmap.png", bbox_inches='tight')
print("✅ Generated Phase 2 Heatmap from CSV Data!")


### 🎥 Phase 3 & 4: Video Upload & Frame Processing
Using OpenCV (`cv2`) to stream video and extract individual frames.

In [ ]:
%%writefile phase4_video_reader.py
import cv2

def inspect_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error opening video stream: {video_path}")
        return
    
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"📹 Video Details:")
    print(f" - Resolution: {width}x{height}")
    print(f" - FPS: {fps:.2f}")
    print(f" - Total Frames: {frame_count}")
    
    ret, frame = cap.read()
    if ret:
        cv2.imwrite("frame_0.jpg", frame)
        print("✅ Saved frame_0.jpg successfully.")
    cap.release()


### 🤖 Phase 5 & 6: Object Tracking & Coordinate Mapping
Map screen coordinates ($1280 \times 720$ px) to pitch space ($120 \times 80$ yards).

In [ ]:
%%writefile tracking_utils.py
def map_pixels_to_pitch(pixel_x, pixel_y, frame_width=1280, frame_height=720, pitch_length=120.0, pitch_width=80.0):
    pitch_x = (pixel_x / frame_width) * pitch_length
    pitch_y = (pixel_y / frame_height) * pitch_width
    
    pitch_x = max(0.0, min(pitch_length, pitch_x))
    pitch_y = max(0.0, min(pitch_width, pitch_y))
    return round(pitch_x, 2), round(pitch_y, 2)


### 🏆 Final Boss Project: Complete Streamlit Dashboard (`app.py`)
Combines vision tracking, dynamic heatmaps, and Gemini AI into one app.

In [ ]:
%%writefile app.py
import streamlit as st
import cv2
import pandas as pd
import numpy as np
import tempfile
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import google.generativeai as genai
from ultralytics import YOLO

st.set_page_config(page_title="Soccer Vision Lab", page_icon="⚽", layout="wide")
st.title("⚽ Soccer Vision Lab: AI Tactical Analyst")
st.write("Upload match footage to track player movement, draw heatmaps, and generate tactical insights.")

target_player_id = st.sidebar.number_input("Target Track ID", min_value=1, value=1, step=1)

try:
    genai.configure(api_key=st.secrets["GEMINI_API_KEY"])
    ai_model = genai.GenerativeModel("gemini-1.5-flash")
except Exception:
    ai_model = None

uploaded_file = st.file_uploader("Upload Video Clip (.mp4)", type=["mp4", "mov"])
if uploaded_file is not None:
    tfile = tempfile.NamedTemporaryFile(delete=False)
    tfile.write(uploaded_file.read())
    cap = cv2.VideoCapture(tfile.name)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    yolo_model = YOLO('yolov8n.pt')
    tracking_records = []
    frame_idx = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx > 150:
            break
        frame_idx += 1
        results = yolo_model.track(frame, persist=True, verbose=False)[0]
        
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.cpu().numpy()
            clss = results.boxes.cls.cpu().numpy()
            
            for box, track_id, cls in zip(boxes, track_ids, clss):
                if int(cls) == 0 and int(track_id) == target_player_id:
                    x1, y1, x2, y2 = box
                    cx, cy = (x1 + x2) / 2.0, y2
                    tracking_records.append({
                        "pitch_x": (cx / frame_width) * 120.0,
                        "pitch_y": (cy / frame_height) * 80.0
                    })
    cap.release()
    
    if tracking_records:
        df_tracking = pd.DataFrame(tracking_records)
        st.subheader("📍 Player Movement Heatmap")
        pitch = Pitch(pitch_type='statsbomb', pitch_color='#aabb97', line_color='white')
        fig, ax = pitch.draw(figsize=(10, 6))
        bin_stat = pitch.bin_statistic(df_tracking['pitch_x'], df_tracking['pitch_y'], statistic='count', bins=(12, 8))
        pitch.heatmap(bin_stat, ax=ax, cmap='Reds', alpha=0.6)
        pitch.scatter(df_tracking['pitch_x'], df_tracking['pitch_y'], c='black', s=20, ax=ax)
        st.pyplot(fig)
